# deepMaze — train DRQN + DTQN (Colab Drive-backed **or** local)

Auto-detects environment via `IS_COLAB` flag set in cell 2:

- **Colab** — mounts Drive, clones the repo into `/content/deepMaze`, persists everything (MLflow, weights, replays) to `${DRIVE_BASE}`.
- **Local** — skips Drive + clone, runs from the existing repo checkout, persists everything under `<repo>/local_runs/`. Bundles also written to `<repo>/assets/` so `python web/server.py` picks them up directly.

**No GCP required either way.**

> **Heads up:** 30×60 with `partial_view=3` is hard — the agent has a 7×7 window in a ~1800-cell maze. Training takes ~1–2 h on T4 for DRQN at 6 k episodes. Drop `MAZE_WIDTH/HEIGHT` (or use the curriculum cell at the bottom) for faster smoke-tests.


## 1 — Mount Drive

In [ ]:
# Detect environment + mount Drive (Colab only).
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    IS_COLAB = False
    print("not on Colab — skipping Drive mount; will use ./local_runs/ for outputs")


## 2 — Config (Colab form)

In [ ]:
# @title Config { run: "auto" }
# ╭─ Repo + paths ──────────────────────────────────────────────────────────╮
REPO_URL    = "https://github.com/juan-garassino/deepMaze.git"  # @param {type:"string"}
REPO_BRANCH = "main"                                             # @param {type:"string"}
DRIVE_BASE  = "/content/drive/MyDrive/deepMaze"                  # @param {type:"string"}

# ╭─ Run ──────────────────────────────────────────────────────────────────╮
AGENTS_TO_RUN     = "drqn,dtqn"  # @param {type:"string"}
RUN_TAG           = "v1"          # @param {type:"string"}
MLFLOW_EXPERIMENT = "deepmaze"   # @param {type:"string"}
SEED              = 0             # @param {type:"integer"}

# NANO_LOCAL=True → when running outside Colab, auto-shrink everything below
# to nano values (8x8 maze, 200 episodes, ~2 min on CPU) — local is for
# pipeline smoke-tests, not convergence. Big training happens on Colab/RunPod.
# Set False to use the full config locally.
NANO_LOCAL = True   # @param {type:"boolean"}

# ╭─ Maze ─────────────────────────────────────────────────────────────────╮
MAZE_WIDTH   = 30    # @param {type:"integer"}
MAZE_HEIGHT  = 60    # @param {type:"integer"}
GENERATOR    = "dfs,random" # @param ["open", "dfs", "random", "dfs,random", "dfs,random,open"]
DENSITY      = 0.2   # @param {type:"number"}
N_TREASURES  = 6     # @param {type:"integer"}
N_LAVA       = 2     # @param {type:"integer"}
COLLECT_ALL  = True  # @param {type:"boolean"}
PARTIAL      = 5     # @param {type:"integer"}
RANDOM_START = True  # @param {type:"boolean"}
BUMP_PENALTY = -0.01 # @param {type:"number"}
# Memory-first: the agent navigates from its recurrent memory, so aux
# observation features default OFF (flip on only for an ablation A/B).
# Shaping stays on — it densifies the reward signal without leaking
# anything into the observation the policy sees at inference.
AUX_FEATURES   = False # @param {type:"boolean"}
REWARD_SHAPING = True  # @param {type:"boolean"}

# ╭─ Periodic eval + curriculum gate ──────────────────────────────────────╮
EVAL_EVERY        = 0    # @param {type:"integer"}   # 0 = num_episodes//10
ADVANCE_THRESHOLD = 0.5  # @param {type:"number"}    # 0 = gate off
STAGE_MAX_REPEATS = 1    # @param {type:"integer"}

# ╭─ Generalization ───────────────────────────────────────────────────────╮
REGENERATE_EVERY = 1     # @param {type:"integer"}
EVAL_REGENERATE  = True  # @param {type:"boolean"}

# ╭─ Training budget ──────────────────────────────────────────────────────╮
NUM_EPISODES  = 3000  # @param {type:"integer"}
MAX_STEPS     = 1620  # @param {type:"integer"}  # ≈ 3*(w+h)*n_treasures for collect_all
EVAL_EPISODES = 50    # @param {type:"integer"}

# ╭─ Agent hyperparameter overrides (0 / 0.0 = use repo default) ──────────╮
# Epsilon decays once per EPISODE (repo default 0.995 → floor ~ep 600).
# For 3000-episode runs 0.998 stretches exploration to ~ep 1500.
# BUFFER_CAPACITY is in EPISODES for drqn/dtqn (repo default 200).
EXPLORATION_DECAY = 0.998  # @param {type:"number"}
BUFFER_CAPACITY   = 0      # @param {type:"integer"}
LEARNING_RATE     = 0.0       # @param {type:"number"}
BATCH_SIZE        = 0         # @param {type:"integer"}
SEQ_LEN           = 0         # @param {type:"integer"}
BURN_IN           = 0         # @param {type:"integer"}
TARGET_SYNC       = 0         # @param {type:"integer"}
LSTM_HIDDEN       = 0         # @param {type:"integer"}

# ╭─ Display + showcase ───────────────────────────────────────────────────╮
PRINT_EVERY     = 25   # @param {type:"integer"}
SHOWCASE_EVERY  = 250  # @param {type:"integer"}
WINDOW          = 100  # @param {type:"integer"}
SHOWCASE_SPRITE = 12   # @param {type:"integer"}
SHOWCASE_FRAMES = 300  # @param {type:"integer"}


## 3 — Clone repo + install deps

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

if IS_COLAB:
    WORK = pathlib.Path("/content/deepMaze")
    os.chdir("/content")
    if WORK.exists():
        shutil.rmtree(WORK)
    subprocess.check_call(["git", "clone", "--depth=1", "-b", REPO_BRANCH, REPO_URL, str(WORK)])
    subprocess.run(["find", str(WORK), "-name", "__pycache__", "-type", "d",
                    "-exec", "rm", "-rf", "{}", "+"], check=False)
    os.chdir(WORK)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "-r", "requirements.txt", "mlflow==2.18.*"])
else:
    # Local: walk up from cwd until we find the repo root marker.
    WORK = pathlib.Path.cwd().resolve()
    while not (WORK / "CLAUDE.md").exists() and WORK.parent != WORK:
        WORK = WORK.parent
    if not (WORK / "CLAUDE.md").exists():
        raise RuntimeError(
            "Could not find repo root (no CLAUDE.md). "
            "Open this notebook from inside the deepMaze repo, or cd into it first."
        )
    os.chdir(WORK)
    # Assume local deps are already installed; if mlflow is missing the next
    # cell will give a clean ImportError.

for sub in ("agents", "environment", "training", "utils", "config"):
    p = str(WORK / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

# Drop cached repo modules so subsequent imports re-read from disk.
_REPO_MODS = ("train", "maze", "render", "viz_events", "recorders", "seeding",
              "hyperparameters", "manager", "replay_buffer", "episode_buffer",
              "report", "visualizations", "session", "bundles", "recurrent",
              "drqn_agent", "dtqn_agent", "dqn_agent",
              "q_agent", "ppo_agent", "base_agent", "nets", "encoders")
for _m in list(sys.modules):
    if _m.split(".")[0] in _REPO_MODS:
        del sys.modules[_m]

if IS_COLAB:
    head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
    print(f"[colab] repo: {head}  branch: {REPO_BRANCH}")
else:
    print(f"[local] repo at {WORK}  (caches cleared)")


## 4 — Drive paths + MLflow tracking URI

In [ ]:
from pathlib import Path

# Local: persist under <repo>/local_runs/ instead of Drive.
DRIVE_BASE_P = Path(DRIVE_BASE) if IS_COLAB else (WORK / "local_runs")
MLRUNS_DIR   = DRIVE_BASE_P / "mlruns"
ASSETS_DIR   = DRIVE_BASE_P / "assets"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = f"file://{MLRUNS_DIR.as_posix()}"

import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# Nano override: when local + NANO_LOCAL, shrink everything so a complete
# train→eval→bundle cycle runs in ~2 min on CPU. Goal is "did the pipeline
# work end-to-end", NOT "did the agent converge". Real training lives on
# Colab (notebook) or RunPod (runpod/Dockerfile).
if not IS_COLAB and NANO_LOCAL:
    print()
    print("⚡ NANO LOCAL OVERRIDE — pipeline smoke-test only (not convergence).")
    print("   Set NANO_LOCAL=False in cell 4 to use your full config locally.")
    MAZE_WIDTH, MAZE_HEIGHT = 8, 8
    N_TREASURES, N_LAVA     = 1, 0
    PARTIAL                 = 2
    COLLECT_ALL             = False
    NUM_EPISODES, MAX_STEPS = 200, 100
    EVAL_EPISODES           = 10
    AGENTS_TO_RUN           = "drqn"
    PRINT_EVERY             = 20
    SHOWCASE_EVERY          = 100
    EXPLORATION_DECAY       = 0.0   # repo default (per-episode 0.995)
    BUFFER_CAPACITY         = 0     # repo default (200 episodes)
    REGENERATE_EVERY        = 1
    # tiny nets so a 2015-era CPU finishes in minutes
    LSTM_HIDDEN             = 32
    BATCH_SIZE              = 4
    SEQ_LEN                 = 4
    EVAL_EPISODES           = 5

print()
print("mode    :", "colab" if IS_COLAB else "local")
print("tracking:", MLFLOW_TRACKING_URI)
print("assets  :", ASSETS_DIR)
print("budget  :", f"{NUM_EPISODES} episodes × max_steps={MAX_STEPS}  ({MAZE_WIDTH}×{MAZE_HEIGHT} maze)")


## 5 — Train one agent (function)

In [ ]:
"""Generic training driver. Reads ALL knobs from cell-4 globals — do not
edit this cell during normal iteration. Tweak cell 4 instead. The actual
session logic lives in training/session.py (shared with RunPod); this cell
only adds the Colab specifics: Drive mirroring + inline image display."""

import shutil
from pathlib import Path

from session import train_session

try:
    from IPython.display import Image as _IPImage, display as _ipy_display
except Exception:
    _IPImage = None
    def _ipy_display(*a, **k): pass


def _agent_overrides(agent_type: str) -> dict:
    candidates = {
        "exploration_decay": EXPLORATION_DECAY,
        "learning_rate":     LEARNING_RATE,
        "batch_size":        BATCH_SIZE,
        "seq_len":           SEQ_LEN,
        "burn_in":           BURN_IN,
        "target_sync":       TARGET_SYNC,
    }
    if agent_type in ("drqn", "dtqn", "dqn"):
        candidates["buffer_capacity"] = BUFFER_CAPACITY
    if agent_type == "drqn":
        candidates["lstm_hidden"] = LSTM_HIDDEN
    return {k: v for k, v in candidates.items() if v}


def train_one(agent_type: str, run_name: str, warm_start_path: str | None = None) -> dict:
    # Showcase dirs: in Colab write to fast SSD and mirror to Drive.
    persistent_showcase = ASSETS_DIR.parent / "showcase" / run_name
    persistent_showcase.mkdir(parents=True, exist_ok=True)
    showcase_dir = (Path(f"/content/showcase/{run_name}")
                    if IS_COLAB else persistent_showcase)

    def _on_showcase(path: Path) -> None:
        if showcase_dir != persistent_showcase:
            try:
                shutil.copy(path, persistent_showcase / path.name)
            except Exception as e:
                print(f"  [showcase] persistent copy failed: {e}")
        if _IPImage is not None:
            _ipy_display(_IPImage(filename=str(path)))

    local_base = Path("assets") if IS_COLAB else (WORK / "assets")
    res = train_session(
        agent_type=agent_type, run_name=run_name,
        env_kw=dict(
            width=MAZE_WIDTH, height=MAZE_HEIGHT, density=DENSITY,
            generator=GENERATOR, n_lava=N_LAVA, n_treasures=N_TREASURES,
            collect_all=COLLECT_ALL, partial_view=PARTIAL, seed=SEED,
            bump_penalty=BUMP_PENALTY,
            aux_features=AUX_FEATURES, reward_shaping=REWARD_SHAPING,
        ),
        num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
        assets_dir=local_base, showcase_dir=showcase_dir,
        agent_overrides=_agent_overrides(agent_type),
        warm_start_path=warm_start_path,
        random_start=RANDOM_START,
        regenerate_every=(REGENERATE_EVERY or None),
        eval_episodes=EVAL_EPISODES, eval_regenerate=EVAL_REGENERATE,
        eval_every=EVAL_EVERY, seed=SEED,
        window=WINDOW, print_every=PRINT_EVERY,
        showcase_every=SHOWCASE_EVERY, showcase_sprite=SHOWCASE_SPRITE,
        showcase_frames=SHOWCASE_FRAMES,
        config_extra=dict(mode="colab" if IS_COLAB else "local"),
        on_showcase=_on_showcase,
        mode_label="colab" if IS_COLAB else "local",
    )

    # Mirror the bundle to the persistent (Drive) location.
    local_out = Path(res["out_dir"])
    persistent_out = ASSETS_DIR / run_name
    if local_out.resolve() != persistent_out.resolve():
        if persistent_out.exists():
            shutil.rmtree(persistent_out)
        shutil.copytree(local_out, persistent_out)
        print(f"  bundle → {persistent_out}")

    res.update(
        drive_path=str(persistent_out),
        showcase_dir=str(persistent_showcase),
        model_path=str(persistent_out / Path(res["model_path"]).name),
        best_model_path=(str(persistent_out / "model.best.pt")
                         if res["best_model_path"] else None),
    )
    return res

## 6 — Train DRQN then DTQN sequentially

In [ ]:
agents = [a.strip() for a in AGENTS_TO_RUN.split(",") if a.strip()]
results = []
for agent_type in agents:
    run_name = f"{agent_type}_{RUN_TAG}"
    results.append(train_one(agent_type, run_name))

print("\n=== summary ===")
for r in results:
    print(f"  {r['agent_type']:5s}  {r['run_name']:20s}  success={r['eval_success_rate']:.2%}  → {r['drive_path']}")


## 7 — Use the bundles locally

On your laptop:

```bash
# Copy a bundle from Drive into the repo's assets/
rsync -a "<google-drive-folder>/deepMaze/assets/drqn_v1/" assets/drqn_v1/

# Then either:
python web/server.py --port 8000    # → http://localhost:8000  → pretrained dropdown
# or:
python main.py --agent_type drqn --resume assets/drqn_v1/model.pt --num_episodes 0
```

To browse MLflow runs locally after copying `mlruns/` down from Drive:

```bash
mlflow ui --backend-store-uri "file://$(pwd)/mlruns"
```


## 8 — Curriculum (optional, alternative to cell 6)

When training on a big maze from scratch fails because the random walker can't stumble on a treasure, **curriculum learning** fixes it: start on a small maze (random exploration finds the goal often), then transfer those weights to a larger maze.

Works here because `PARTIAL` keeps the observation shape constant — the network sees an 11×11 window whether the underlying maze is 10×20 or 30×60 — so weights load cleanly between stages.

Run `results = run_curriculum("drqn")` (or `"dtqn"`) in the next cell. Edit `CURRICULUM_STAGES` to change the schedule.


In [ ]:
# @title Curriculum schedule { run: "auto" }
# Each stage: (width, height, n_treasures, num_episodes, max_steps, seq_len).
# seq_len scales the memory window with the maze (burn_in follows as
# seq_len//2); it is not shape-bearing, so weight transfer still works.
# Reasonable for DRQN on a T4 GPU.
# Collect-all tours: max_steps = 3*(w+h)*n_treasures; treasure counts ramp
# 2/4/6 so keep-going-after-pickup is learned early. The revisit_rate in the
# periodic eval line is the memory proof — it should FALL as memory forms.
CURRICULUM_STAGES = [
    (10, 20, 2,  800, 180, 8),
    (20, 40, 4, 1500, 720, 16),
    (30, 60, 6, 2500, 1620, 32),
]


def run_curriculum(agent_type: str, run_tag: str = "curr") -> list[dict]:
    """Train through CURRICULUM_STAGES with weight transfer between stages."""
    global MAZE_WIDTH, MAZE_HEIGHT, N_TREASURES, NUM_EPISODES, MAX_STEPS
    global SEQ_LEN, BURN_IN
    saved = (MAZE_WIDTH, MAZE_HEIGHT, N_TREASURES, NUM_EPISODES, MAX_STEPS,
             SEQ_LEN, BURN_IN)

    results = []
    warm = None
    try:
        for i, stage in enumerate(CURRICULUM_STAGES):
            w, h, nt, neps, mx = stage[:5]
            sq = stage[5] if len(stage) > 5 else 0
            MAZE_WIDTH, MAZE_HEIGHT = w, h
            N_TREASURES = nt
            NUM_EPISODES, MAX_STEPS = neps, mx
            if sq:
                SEQ_LEN, BURN_IN = sq, max(2, sq // 2)

            res = None
            for attempt in range(1 + max(0, STAGE_MAX_REPEATS - 1)):
                run_name = f"{agent_type}_{run_tag}_s{i}_{w}x{h}" + (
                    f"_r{attempt}" if attempt else "")
                print()
                print("█" * 78)
                print(f"  CURRICULUM stage {i+1}/{len(CURRICULUM_STAGES)}  —  "
                      f"{w}x{h}  ({nt} treasures, {neps} eps, max_steps={mx})")
                if warm:
                    print(f"  warm-starting from: {warm}")
                print("█" * 78)

                res = train_one(agent_type, run_name, warm_start_path=warm)
                results.append(res)
                gate = max(res["eval_success_rate"], res["best_success_rate"])
                if not ADVANCE_THRESHOLD or gate >= ADVANCE_THRESHOLD:
                    break
                warm = res["best_model_path"] or res["model_path"]
                print(f"  ⟳ stage below gate ({gate:.1%} < "
                      f"{ADVANCE_THRESHOLD:.1%}) — retrying from best")
            gate = max(res["eval_success_rate"], res["best_success_rate"])
            if ADVANCE_THRESHOLD and gate < ADVANCE_THRESHOLD:
                print(f"  ✗ stage {i} never passed the gate — stopping curriculum")
                break
            warm = res["best_model_path"] or res["model_path"]
    finally:
        (MAZE_WIDTH, MAZE_HEIGHT, N_TREASURES, NUM_EPISODES, MAX_STEPS,
         SEQ_LEN, BURN_IN) = saved

    print()
    print("=" * 78)
    print("  CURRICULUM SUMMARY")
    print("=" * 78)
    for r in results:
        print(f"  {r['run_name']:40s}  success={r['eval_success_rate']:.2%}")
    return results


# Uncomment to run:
# results = run_curriculum("drqn")
# results = run_curriculum("dtqn", run_tag="curr_dtqn")
